# Example: Extracting Information

Explore instrument metadata, executable bid/ask prices, basket valuation, and account state through read-only methods. This notebook uses the synthetic exchange so it runs without credentials.


In [1]:
from pathlib import Path
import os
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "bot.py").is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Project root located.")


Project root located.


In [2]:
from helper import load_demo
exchange, config = load_demo()
instruments = exchange.get_instruments()
for instrument_id, metadata in instruments.items():
    print(instrument_id, "tick:", metadata.tick_size, "paused:", metadata.paused)


SEMIS_ETF_US tick: 0.1 paused: False
SEMIS_ETF_EU tick: 0.1 paused: False
CHIP_A tick: 0.1 paused: False
CHIP_B tick: 0.1 paused: False
CHIP_C tick: 0.1 paused: False


## Best bid, best ask, spread, and displayed size

In [3]:
rows = []
for instrument_id in instruments:
    book = exchange.get_last_price_book(instrument_id)
    if book and book.bids and book.asks:
        bid, ask = book.bids[0], book.asks[0]
        rows.append({"instrument": instrument_id, "bid": bid.price, "bid_volume": bid.volume,
                     "ask": ask.price, "ask_volume": ask.volume, "spread": round(ask.price - bid.price, 6)})
for row in rows:
    print(row)


{'instrument': 'SEMIS_ETF_US', 'bid': 106.9, 'bid_volume': 50, 'ask': 107.1, 'ask_volume': 50, 'spread': 0.2}
{'instrument': 'SEMIS_ETF_EU', 'bid': 107.9, 'bid_volume': 50, 'ask': 108.1, 'ask_volume': 50, 'spread': 0.2}
{'instrument': 'CHIP_A', 'bid': 99.9, 'bid_volume': 50, 'ask': 100.1, 'ask_volume': 50, 'spread': 0.2}
{'instrument': 'CHIP_B', 'bid': 109.9, 'bid_volume': 50, 'ask': 110.1, 'ask_volume': 50, 'spread': 0.2}
{'instrument': 'CHIP_C', 'bid': 119.9, 'bid_volume': 50, 'ask': 120.1, 'ask_volume': 50, 'spread': 0.2}


## Basket midpoint and executable bounds

In [4]:
nav = config["cash_component"]
sell_basket = config["cash_component"]
buy_basket = config["cash_component"]
for stock, weight in config["weights"].items():
    book = exchange.get_last_price_book(stock)
    nav += weight * (book.bids[0].price + book.asks[0].price) / 2
    sell_basket += weight * book.bids[0].price
    buy_basket += weight * book.asks[0].price
print("Midpoint NAV:", round(nav, 4))
print("Sell-basket proceeds before fees:", round(sell_basket, 4))
print("Buy-basket cost before fees:", round(buy_basket, 4))


Midpoint NAV: 107.0
Sell-basket proceeds before fees: 106.9
Buy-basket cost before fees: 107.1


## Account information and an invariant: no orders were sent

In [5]:
print("Positions:", exchange.get_positions())
print("PNL:", exchange.get_pnl())
assert exchange.counter == 0
assert not exchange.fills
print("Read-only inspection confirmed: zero orders and zero fills.")


Positions: {}
PNL: 0.0
Read-only inspection confirmed: zero orders and zero fills.


## Official SDK counterpart

The official SDK exposes equivalent discovery methods, but requires competition access. `python3 example1.py --engine optibook` provides a read-only CLI. The reference repository bundles its SDK; this project documents setup in `optibook/README.md` instead of copying that client.
